<a href="https://colab.research.google.com/github/ksuplee/AI_Agent/blob/main/10_3_Application_of_Retriever%E2%80%93Generator_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[실습 10-3] Retriever–Generator 파이프라인 응용  

### 실습 목표

- 사용자가 업로드한 문서(PDF)를 실시간으로 처리하여 **지식 베이스(Knowledge Base)**로 변환할 수 있다 [cite: 10-1-2].  

**Retriever(검색기)**와 **Generator(생성기)**를 LCEL(LangChain Expression Language)로 연결한 파이프라인을 구축한다 [cite: 10-3-3].  

검색된 문서의 내용과 **출처(Source/Page)**를 함께 제시하는 신뢰성 있는 응답 구조를 구현한다 [cite: 10-1-1].  

---

### RAG 파이프라인의 4단계  

- Indexing (색인): PDF를 불러와 텍스트를 쪼개고(Split), 벡터로 변환(Embed)하여 DB에 저장한다 [cite: 10-1-2].  

- Retrieval (검색): 사용자의 질문과 가장 유사한 문서 조각을 벡터 DB에서 찾아낸다 [cite: 10-1-2].  

- Augmentation (증강): 질문에 검색된 지식(Context)을 덧붙여 풍부한 배경지식을 만든다 [cite: 10-1-2].  

- Generation (생성): LLM이 제공된 지식 범위 안에서 정답과 출처를 생성한다 [cite: 10-1-2].  

---


1. 환경 준비 및 라이브러리 설치  

- RAG 구현을 위해 PDF 파싱, 임베딩, 벡터 검색 라이브러리를 설치합니다.  

In [1]:
# 1. 필수 라이브러리 설치
!pip install -q -U langchain langchain-community langchain-google-genai pypdf faiss-cpu google-generativeai


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.7/111.7 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 330.6/330.6 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 500.1/500.1 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 158.1/158.1 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests=

In [2]:
# Google API Key 설정
import google.genai as genai # Recommended: Use the newer google.genai
from google.colab import userdata

GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
# genai.configure(api_key=GOOGLE_API_KEY) # Not needed for google.genai, and can cause conflicts

print("Gemini API 설정 완료")

Gemini API 설정 완료


In [3]:
import google.generativeai as genai
from google.colab import userdata # GOOGLE_API_KEY를 가져오기 위해 추가

GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY') # API 키를 다시 가져옴
genai.configure(api_key=GOOGLE_API_KEY) # genai 클라이언트 구성

print("사용 가능한 임베딩 모델 목록:")
for m in genai.list_models():
    if "embedContent" in m.supported_generation_methods:
        print(m.name)

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


사용 가능한 임베딩 모델 목록:
models/gemini-embedding-001


2. 실습: 문서 업로드 및 검색 기반 응답 구현  

- Google Colab 환경에서 작동하는 완성형 RAG 에이전트 코드입니다.  

In [4]:
# 1. 필수 라이브러리 설치
# !pip install -q -U langchain langchain-community langchain-google-genai pypdf faiss-cpu

import google.genai as genai
from google.colab import userdata
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from google.colab import files

# --- [단계 1: Indexing] 문서 업로드 및 벡터 DB 구축 ---
def create_retriever(pdf_path):
    loader = PyPDFLoader(pdf_path)
    # 문서 로드 시 각 페이지의 텍스트와 페이지 번호가 메타데이터로 저장됨
    documents = loader.load()

    # 텍스트 분할: 검색 품질을 위해 1000자 단위로 쪼개고 100자 중첩
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
    splits = text_splitter.split_documents(documents)

    # 벡터화 (Gemini Embedding 모델 사용)
    # GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
    # embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-001", google_api_key=GOOGLE_API_KEY)
    # 2. 임베딩 생성 및 벡터 DB 저장 (Indexing) [cite: 10-1-2, 10-3-3]
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
    embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001", google_api_key=GOOGLE_API_KEY)

    # FAISS 벡터 DB 생성
    vectorstore = FAISS.from_documents(documents=splits, embedding=embeddings)
    return vectorstore.as_retriever(search_kwargs={"k": 3}) # 관련 조각 3개 추출

# --- [단계 2: Pipeline] Retriever + Generator 연결 ---
def build_rag_chain(retriever):
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
    llm = ChatGoogleGenerativeAI(model="gemini-flash-latest", google_api_key=GOOGLE_API_KEY)

    # 출처 명시를 유도하는 프롬프트 설계
    template = """아래 제공된 [컨텍스트]를 사용하여 질문에 답하세요.
    답변 시 반드시 정보가 위치한 페이지 번호를 함께 명시하세요.
    내용에 정답이 없으면 지어내지 말고 모른다고 하세요.

    [컨텍스트]
    {context}

    질문: {question}
    답변:"""

    prompt = ChatPromptTemplate.from_template(template)

    # LCEL 파이프라인 구축
    chain = (
        {"context": retriever, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )
    return chain

# --- [단계 3: 실행] ---
uploaded = files.upload()

for fn in uploaded.keys():
    print('User uploaded file "{name}" with length {length} bytes'.format(
        name=fn, length=len(uploaded[fn])))
    pdf_path = f"/content/{fn}"

print(f"Uploaded PDF path: {pdf_path}")
# pdf_path = "/content/your_document.pdf"
retriever = create_retriever(pdf_path)
rag_agent = build_rag_chain(retriever)
print(rag_agent.invoke("본 법령의 제7조의 내용은 무엇인가요?"))

Saving 인공지능 발전과 신뢰 기반 조성 등에 관한 기본법(법률)(제21311호)(20260122).pdf to 인공지능 발전과 신뢰 기반 조성 등에 관한 기본법(법률)(제21311호)(20260122).pdf
User uploaded file "인공지능 발전과 신뢰 기반 조성 등에 관한 기본법(법률)(제21311호)(20260122).pdf" with length 170010 bytes
Uploaded PDF path: /content/인공지능 발전과 신뢰 기반 조성 등에 관한 기본법(법률)(제21311호)(20260122).pdf
본 법령의 제7조는 **국가인공지능전략위원회**의 설치 및 구성에 관한 내용을 담고 있으며, 주요 내용은 다음과 같습니다.

1.  **설치 및 목적**: 인공지능 발전과 신뢰 기반 조성 등을 위한 주요 정책 등을 심의·의결하기 위하여 대통령 소속으로 국가인공지능전략위원회를 둡니다. (4페이지)
2.  **구성**: 위원회는 위원장 1명과 3명 이내의 부위원장을 포함한 60명 이내의 위원으로 구성됩니다. (4페이지)
3.  **비밀 누설 금지 및 벌칙**: 제7조제9항을 위반하여 직무상 알게 된 비밀을 타인에게 누설하거나 직무상 목적 외의 용도로 사용한 자는 3년 이하의 징역 또는 3천만원 이하의 벌금에 처합니다. (16페이지)


3. 핵심 분석: 왜 이렇게 동작하는가?  

- 데이터 증강(Augmentation): retriever가 벡터 DB에서 찾아온 텍스트 조각들이 {context} 변수에 자동으로 주입됩니다 [cite: 10-1-2].  

- 할루시네이션 억제: 모델의 내부 기억이 아니라 주입된 텍스트 범위 내에서만 답변하도록 프롬프트로 강제합니다 [cite: 10-1-1, 10-3-3].  

- 메타데이터 활용: PyPDFLoader가 추출한 페이지 번호 정보를 통해 답변에 신뢰도를 더합니다 [cite: 10-1-2].  

4. 단순 요약 vs RAG 에이전트 비교 [cite: 10-1-3]  

| 구분 | 단순 PDF 요약 (실습 10-2) | RAG 에이전트 (실습 10-3) |
|------|---------------------------|---------------------------|
| 처리 용량 | 토큰 제한으로 긴 문서 처리 불가 | 수만 페이지의 문서도 처리 가능 |
| 답변 방식 | 전체 내용을 뭉뚱그려 설명 | 질문과 관련된 특정 구절을 찾아 상세 답변 |
| 정확도 | 중요한 세부 사항이 누락될 수 있음 | 검색된 원문을 근거로 하므로 매우 정밀함 |
